In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.webdriver import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import requests
import os
import time
import csv

In [16]:
# Set up Chrome WebDriver
service = Service(ChromeDriverManager().install())
options = Options()
driver = webdriver.Chrome(service=service, options=options)

# Target URL
url = "https://www.tweedekamer.nl/kamerstukken/stemmingsuitslagen/detail?id=2025P04578&did=2025P04578"
driver.get(url)

# Prepare list to store extracted data
data = []

try:
    # Locate all <p> elements with class "u-mt-8"
    p_elements = driver.find_elements(By.CSS_SELECTOR, "p.u-mt-8")

    for p_element in p_elements:
        try:
            # Extract text from <span class="u-font-bold">
            span_element = p_element.find_element(By.CSS_SELECTOR, "span.u-font-bold")
            extracted_text = span_element.text.strip()
        except:
            continue  # Skip if the <span> element is not found

        # Process only if the text is "Aangenomen." or "Verworpen."
        if extracted_text in ["Aangenomen.", "Verworpen."]:
            href_value = "N/A"
            try:
                h3_element = driver.find_element(By.CSS_SELECTOR, "div.m-card__main div.m-card__content div.t-grid div.t-grid__col h3.m-card__title")
                a_element = h3_element.find_element(By.TAG_NAME, "a")
                href_value = a_element.get_attribute("href")
            except:
                pass  # If no link is found, we leave it as "N/A"

            # Append extracted data to list
            data.append([extracted_text, href_value])

except Exception as e:
    print("Error:", e)

finally:
    driver.quit()

# Save extracted data to a CSV file only if there is data to save
if data:
    csv_filename = "extracted_data.csv"
    with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["Text", "Href"])  # CSV headers
        writer.writerows(data)

    print(f"Data saved to {csv_filename}")
else:
    print("No relevant data found. No CSV file was created.")

Data saved to extracted_data.csv


In [ ]:
# Setting up WebDriver with optimisations
service = Service(ChromeDriverManager().install())
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-gpu")
# Disabling images and extensions to speed up the code
options.add_argument("--blink-settings=imagesEnabled=false")
options.add_argument("--disable-extensions")

driver = webdriver.Chrome(service=service, options=options)

# Setting out target URL
url = "https://www.tweedekamer.nl/kamerstukken/detail?id=2025Z05082&did=2025D11726"
driver.get(url)

try:
    # Waiting for the header to extract its text
    h1_element = WebDriverWait(driver, 5).until(EC.visibility_of_element_located((By.TAG_NAME, "h1")))
    motion_text = h1_element.text.strip()
    print("\nMotion content:\n", motion_text)

    # Locate the petitioner section efficiently
    petition_section = driver.find_elements(By.CSS_SELECTOR, "div.u-mt-8 ul.m-list span.m-list__label")
    petitioner_texts = [item.text.strip() for item in petition_section]
    
    print("\nPetitioner:\n", "\n".join(petitioner_texts))

    # Click the button to reveal the voting table
    button = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button.m-toggler__handler")))
    driver.execute_script("arguments[0].click();", button)

    # Wait for the table to appear
    tbody = WebDriverWait(driver, 5).until(EC.visibility_of_element_located((By.CSS_SELECTOR, "div.m-toggler__body div.u-overflow-x-auto table.h-table-bordered tbody")))

    # Extract all rows with <td> (skip rows with only <th>)
    extracted_data = []
    for row in tbody.find_elements(By.TAG_NAME, "tr"):
        columns = row.find_elements(By.TAG_NAME, "td")
        if columns:  # Only process rows that contain <td>
            extracted_data.append([col.text.strip() for col in columns])

    # Print the extracted voting results
    print("\nVoting results:")
    for row in extracted_data:
        print(row)

except Exception as e:
    print("Error:", e)

finally:
    driver.quit()


Motion content:
 Motie
:
Motie van de leden Bushoff en Westerveld over artsen en zorgpersoneel niet laten opdraaien voor de bezuiniging op de SOV

Petitioner:
 Indiener
Julian Bushoff, Kamerlid
Medeindiener
Lisa Westerveld, Kamerlid

Voting results:
['PVV', '37', 'Tegen']
['GroenLinks-PvdA', '25', 'Voor']
['VVD', '24', 'Tegen']
['NSC', '20', 'Tegen']
['D66', '9', 'Voor']
['BBB', '7', 'Tegen']
['CDA', '5', 'Voor']
['SP', '5', 'Voor']
['ChristenUnie', '3', 'Voor']
['DENK', '3', 'Voor']
['FVD', '3', 'Voor']
['PvdD', '3', 'Voor']
['SGP', '3', 'Voor']
['Volt', '2', 'Voor']
['JA21', '1', 'Tegen']
